# Phase 5.7: Inference & Visualization for VGGT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase5/07_inference_visualization.ipynb)

## Learning Objectives
- 理解 VGGT 的完整推理流程
- 掌握深度图和置信度的可视化方法
- 学习相机位姿和内参的 3D 可视化
- 实现多视图点云融合和可视化
- 理解 Track 可视化和轨迹追踪
- 分析推理性能和内存需求

## Estimated Time: 60 minutes

## Environment Setup

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import pandas as pd
import seaborn as sns

# 设置随机种子
np.random.seed(42)

# 设置绘图风格
plt.style.use('default')
sns.set_palette("husl")

print("Environment setup complete!")

## 1. Inference Pipeline Overview

VGGT 的推理流程包括以下步骤：

1. **Image Preprocessing**: 调整大小、归一化
2. **Encoder**: DINOv2 特征提取
3. **Aggregator**: 跨视图特征融合
4. **Multiple Heads**: 并行预测深度、相机、点云、Track

### GPU Memory Requirements

| Image Count | Resolution | bfloat16 | float16 | float32 |
|-------------|-----------|----------|---------|----------|
| 2 views     | 512×512   | 8 GB     | 10 GB   | 16 GB    |
| 4 views     | 512×512   | 12 GB    | 16 GB   | 28 GB    |
| 8 views     | 512×512   | 20 GB    | 28 GB   | 48 GB    |
| 16 views    | 512×512   | 36 GB    | 52 GB   | 92 GB    |

### bfloat16 vs float16
- **bfloat16**: 更好的数值稳定性，推荐用于训练和推理
- **float16**: 更高的精度，但可能出现溢出
- VGGT 在 H100 上默认使用 bfloat16

In [ ]:
# 可视化推理流程
def visualize_inference_pipeline():
    """
    可视化 VGGT 的完整推理流程
    """
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis('off')
    
    # 定义颜色方案
    colors = {
        'input': '#E8F4F8',
        'encoder': '#B3E5FC',
        'aggregator': '#81D4FA',
        'heads': '#4FC3F7',
        'output': '#29B6F6'
    }
    
    # 1. Input Images
    input_box = FancyBboxPatch((1, 10), 3, 1.2, 
                               boxstyle="round,pad=0.1", 
                               edgecolor='black', 
                               facecolor=colors['input'], 
                               linewidth=2)
    ax.add_patch(input_box)
    ax.text(2.5, 10.6, 'Input Images\n[B,S,3,H,W]', 
            ha='center', va='center', fontsize=11, fontweight='bold')
    
    # Arrow down
    ax.annotate('', xy=(2.5, 9.8), xytext=(2.5, 9.2),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    # 2. DINOv2 Encoder
    encoder_box = FancyBboxPatch((0.5, 7.5), 4, 1.5, 
                                 boxstyle="round,pad=0.1", 
                                 edgecolor='black', 
                                 facecolor=colors['encoder'], 
                                 linewidth=2)
    ax.add_patch(encoder_box)
    ax.text(2.5, 8.25, 'DINOv2 Encoder\n(Frozen)', 
            ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(2.5, 7.85, '[B,S,N,D]', ha='center', va='center', fontsize=9)
    
    # Arrow down
    ax.annotate('', xy=(2.5, 7.3), xytext=(2.5, 6.7),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    # 3. Aggregator
    agg_box = FancyBboxPatch((0.5, 5.0), 4, 1.5, 
                             boxstyle="round,pad=0.1", 
                             edgecolor='black', 
                             facecolor=colors['aggregator'], 
                             linewidth=2)
    ax.add_patch(agg_box)
    ax.text(2.5, 5.75, 'Aggregator\n(Cross-view Attention)', 
            ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(2.5, 5.35, '[B,S,N,D]', ha='center', va='center', fontsize=9)
    
    # Arrows to heads
    for i, x in enumerate([1, 3, 5, 7]):
        ax.annotate('', xy=(x+0.75, 3.8), xytext=(2.5, 4.8),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
    
    # 4. Prediction Heads
    heads = [
        ('Depth Head', '[B,S,H,W,1]', 0.5),
        ('Camera Head', '[B,S,3,4]\n[B,S,2]', 2.5),
        ('Point Head', '[B,S,H,W,3]', 4.5),
        ('Track Head', '[B,Q,S,2]', 6.5)
    ]
    
    for i, (name, shape, x) in enumerate(heads):
        head_box = FancyBboxPatch((x, 2.5), 1.8, 1.2, 
                                  boxstyle="round,pad=0.08", 
                                  edgecolor='black', 
                                  facecolor=colors['heads'], 
                                  linewidth=2)
        ax.add_patch(head_box)
        ax.text(x+0.9, 3.3, name, ha='center', va='center', 
                fontsize=9, fontweight='bold')
        ax.text(x+0.9, 2.85, shape, ha='center', va='center', fontsize=7)
        
        # Arrow down
        ax.annotate('', xy=(x+0.9, 2.3), xytext=(x+0.9, 1.7),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
    
    # 5. Outputs
    outputs = [
        ('Depth Map', 0.5),
        ('Camera Pose', 2.5),
        ('Point Cloud', 4.5),
        ('Tracks', 6.5)
    ]
    
    for name, x in outputs:
        output_box = FancyBboxPatch((x, 0.5), 1.8, 1.0, 
                                    boxstyle="round,pad=0.08", 
                                    edgecolor='black', 
                                    facecolor=colors['output'], 
                                    linewidth=2)
        ax.add_patch(output_box)
        ax.text(x+0.9, 1.0, name, ha='center', va='center', 
                fontsize=9, fontweight='bold')
    
    plt.title('VGGT Inference Pipeline', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

visualize_inference_pipeline()

## 2. Running VGGT Step-by-Step

模拟完整的推理流程，包括：
- 输入预处理
- 特征提取
- 特征聚合
- 多头预测

使用合成数据演示形状变化。

In [ ]:
# 模拟 VGGT 推理流程
def simulate_vggt_inference(batch_size=2, num_views=4, img_size=512):
    """
    模拟 VGGT 的完整推理流程
    
    Args:
        batch_size: 批次大小
        num_views: 视图数量
        img_size: 图像尺寸
    """
    print("=" * 60)
    print("VGGT Inference Pipeline Simulation")
    print("=" * 60)
    
    B, S, H, W = batch_size, num_views, img_size, img_size
    
    # 1. Input Images
    print("\n[Step 1] Input Images")
    images = np.random.randn(B, S, 3, H, W).astype(np.float32)
    print(f"  Shape: {images.shape}")
    print(f"  Memory: {images.nbytes / 1024**2:.2f} MB")
    
    # 2. Preprocessing (Resize to 518x518 for DINOv2)
    print("\n[Step 2] Preprocessing")
    dino_size = 518  # DINOv2 输入尺寸
    images_resized = np.random.randn(B, S, 3, dino_size, dino_size).astype(np.float32)
    print(f"  Resized Shape: {images_resized.shape}")
    print(f"  Normalized: mean=0, std=1")
    
    # 3. DINOv2 Encoder
    print("\n[Step 3] DINOv2 Encoder")
    patch_size = 14
    num_patches = (dino_size // patch_size) ** 2
    feat_dim = 1024  # DINOv2-large
    encoder_features = np.random.randn(B, S, num_patches, feat_dim).astype(np.float32)
    print(f"  Feature Shape: {encoder_features.shape}")
    print(f"  Num Patches: {num_patches} = ({dino_size}/{patch_size})^2")
    print(f"  Feature Dim: {feat_dim}")
    print(f"  Memory: {encoder_features.nbytes / 1024**2:.2f} MB")
    
    # 4. Aggregator (Cross-view Attention)
    print("\n[Step 4] Aggregator")
    aggregated_features = np.random.randn(B, S, num_patches, feat_dim).astype(np.float32)
    print(f"  Aggregated Shape: {aggregated_features.shape}")
    print(f"  Cross-view attention applied across {S} views")
    print(f"  Memory: {aggregated_features.nbytes / 1024**2:.2f} MB")
    
    # 5. Depth Head
    print("\n[Step 5] Depth Head")
    depth_h, depth_w = H // 8, W // 8  # 下采样 8x
    depth_map = np.random.randn(B, S, depth_h, depth_w, 1).astype(np.float32)
    depth_confidence = np.random.rand(B, S, depth_h, depth_w).astype(np.float32)
    print(f"  Depth Map Shape: {depth_map.shape}")
    print(f"  Depth Confidence Shape: {depth_confidence.shape}")
    print(f"  Activation: exp(depth)")
    print(f"  Memory: {(depth_map.nbytes + depth_confidence.nbytes) / 1024**2:.2f} MB")
    
    # 6. Camera Head
    print("\n[Step 6] Camera Head")
    extrinsics = np.random.randn(B, S, 3, 4).astype(np.float32)  # [R|t]
    fov = np.random.rand(B, S, 2).astype(np.float32) * 60 + 30  # 30-90 degrees
    print(f"  Extrinsics Shape: {extrinsics.shape}")
    print(f"  FoV Shape: {fov.shape}")
    print(f"  Extrinsics: [R|t] format (rotation + translation)")
    print(f"  FoV: [fov_x, fov_y] in degrees")
    print(f"  Memory: {(extrinsics.nbytes + fov.nbytes) / 1024**2:.2f} MB")
    
    # 7. Point Head
    print("\n[Step 7] Point Head")
    point_h, point_w = H // 8, W // 8
    points = np.random.randn(B, S, point_h, point_w, 3).astype(np.float32)
    point_confidence = np.random.rand(B, S, point_h, point_w).astype(np.float32)
    print(f"  Points Shape: {points.shape}")
    print(f"  Point Confidence Shape: {point_confidence.shape}")
    print(f"  Memory: {(points.nbytes + point_confidence.nbytes) / 1024**2:.2f} MB")
    
    # 8. Track Head
    print("\n[Step 8] Track Head")
    num_queries = 1024
    tracks = np.random.rand(B, num_queries, S, 2).astype(np.float32) * img_size
    track_visibility = np.random.rand(B, num_queries, S).astype(np.float32)
    print(f"  Tracks Shape: {tracks.shape}")
    print(f"  Track Visibility Shape: {track_visibility.shape}")
    print(f"  Num Queries: {num_queries}")
    print(f"  Memory: {(tracks.nbytes + track_visibility.nbytes) / 1024**2:.2f} MB")
    
    # Total Memory
    total_memory = (
        images.nbytes + 
        encoder_features.nbytes + 
        aggregated_features.nbytes +
        depth_map.nbytes + depth_confidence.nbytes +
        extrinsics.nbytes + fov.nbytes +
        points.nbytes + point_confidence.nbytes +
        tracks.nbytes + track_visibility.nbytes
    ) / 1024**2
    
    print("\n" + "=" * 60)
    print(f"Total Memory (Activations Only): {total_memory:.2f} MB")
    print(f"Estimated GPU Memory (with model): ~{total_memory * 2:.2f} MB")
    print("=" * 60)
    
    return {
        'depth': (depth_map, depth_confidence),
        'camera': (extrinsics, fov),
        'points': (points, point_confidence),
        'tracks': (tracks, track_visibility)
    }

# 运行模拟
outputs = simulate_vggt_inference(batch_size=2, num_views=4, img_size=512)

## 3. Visualizing Depth Maps

深度图可视化是 VGGT 推理结果的重要部分：

- **Depth Output**: `[B, S, H, W, 1]` 经过 `exp` 激活
- **Confidence Maps**: `[B, S, H, W]` 表示预测置信度
- **Color Mapping**: viridis, inferno, turbo 等

我们将创建合成深度图并使用多种配色方案可视化。

In [ ]:
# 创建合成深度图
def create_synthetic_depth_maps(num_views=4, height=128, width=128):
    """
    创建合成深度图和置信度图
    """
    depths = []
    confidences = []
    
    for i in range(num_views):
        # 创建不同的深度模式
        x = np.linspace(-2, 2, width)
        y = np.linspace(-2, 2, height)
        X, Y = np.meshgrid(x, y)
        
        # 不同视图的深度变化
        if i % 4 == 0:
            # 平面
            depth = 5.0 + 0.5 * X
        elif i % 4 == 1:
            # 球面
            depth = 5.0 + np.sqrt(X**2 + Y**2)
        elif i % 4 == 2:
            # 波浪
            depth = 5.0 + np.sin(X) * np.cos(Y)
        else:
            # 双峰
            depth = 5.0 + np.exp(-((X-1)**2 + Y**2)) + np.exp(-((X+1)**2 + Y**2))
        
        # 添加噪声
        depth += np.random.randn(height, width) * 0.1
        
        # 创建置信度图 (边缘置信度较低)
        confidence = np.ones((height, width))
        confidence[:10, :] *= 0.3
        confidence[-10:, :] *= 0.3
        confidence[:, :10] *= 0.3
        confidence[:, -10:] *= 0.3
        confidence += np.random.rand(height, width) * 0.2
        confidence = np.clip(confidence, 0, 1)
        
        depths.append(depth)
        confidences.append(confidence)
    
    return np.array(depths), np.array(confidences)

# 可视化深度图
def visualize_depth_maps(depths, confidences, colormaps=['viridis', 'inferno', 'turbo']):
    """
    使用多种配色方案可视化深度图
    """
    num_views = len(depths)
    num_colormaps = len(colormaps)
    
    fig, axes = plt.subplots(num_views, num_colormaps + 1, 
                            figsize=(4 * (num_colormaps + 1), 4 * num_views))
    
    if num_views == 1:
        axes = axes[np.newaxis, :]
    
    for i in range(num_views):
        depth = depths[i]
        confidence = confidences[i]
        
        # 绘制置信度图
        im = axes[i, 0].imshow(confidence, cmap='gray', vmin=0, vmax=1)
        axes[i, 0].set_title(f'View {i+1}: Confidence', fontsize=12, fontweight='bold')
        axes[i, 0].axis('off')
        plt.colorbar(im, ax=axes[i, 0], fraction=0.046, pad=0.04)
        
        # 绘制不同配色的深度图
        for j, cmap in enumerate(colormaps):
            im = axes[i, j+1].imshow(depth, cmap=cmap)
            axes[i, j+1].set_title(f'View {i+1}: Depth ({cmap})', 
                                  fontsize=12, fontweight='bold')
            axes[i, j+1].axis('off')
            plt.colorbar(im, ax=axes[i, j+1], fraction=0.046, pad=0.04)
    
    plt.suptitle('Multi-View Depth Map Visualization', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

# 创建并可视化深度图
depths, confidences = create_synthetic_depth_maps(num_views=4, height=128, width=128)
visualize_depth_maps(depths, confidences, colormaps=['viridis', 'inferno', 'turbo'])

## 4. Visualizing Camera Poses

相机位姿可视化帮助理解多视图几何：

- **Extrinsics**: `[B, S, 3, 4]` 格式 `[R|t]`
- **Camera Frustum**: 3D 空间中的相机视锥
- **Trajectory**: 相机轨迹

我们将实现相机视锥的 3D 可视化。

In [ ]:
# 生成相机位姿
def generate_camera_poses(num_views=8, radius=5.0):
    """
    生成圆形轨迹的相机位姿
    
    Args:
        num_views: 视图数量
        radius: 轨迹半径
    
    Returns:
        extrinsics: [S, 3, 4] 相机外参
        positions: [S, 3] 相机位置
    """
    extrinsics = []
    positions = []
    
    for i in range(num_views):
        # 圆形轨迹
        angle = 2 * np.pi * i / num_views
        
        # 相机位置
        cam_pos = np.array([
            radius * np.cos(angle),
            radius * np.sin(angle),
            2.0 + 0.5 * np.sin(2 * angle)  # 添加高度变化
        ])
        
        # 相机朝向原点
        forward = -cam_pos / np.linalg.norm(cam_pos)
        up = np.array([0, 0, 1])
        right = np.cross(forward, up)
        right = right / np.linalg.norm(right)
        up = np.cross(right, forward)
        
        # 构建旋转矩阵
        R = np.stack([right, up, -forward], axis=0)
        
        # 构建外参矩阵 [R|t]
        extrinsic = np.concatenate([R, cam_pos[:, np.newaxis]], axis=1)
        
        extrinsics.append(extrinsic)
        positions.append(cam_pos)
    
    return np.array(extrinsics), np.array(positions)

# 绘制相机视锥
def draw_camera_frustum(ax, extrinsic, fov_deg=60, frustum_size=1.0, color='blue'):
    """
    绘制相机视锥
    
    Args:
        ax: matplotlib 3D axis
        extrinsic: [3, 4] 相机外参
        fov_deg: 视场角 (度)
        frustum_size: 视锥大小
        color: 视锥颜色
    """
    R = extrinsic[:, :3]
    t = extrinsic[:, 3]
    
    # 计算视锥顶点
    fov_rad = np.deg2rad(fov_deg)
    half_size = frustum_size * np.tan(fov_rad / 2)
    
    # 相机坐标系中的视锥顶点
    corners_cam = np.array([
        [0, 0, 0],  # 相机中心
        [-half_size, -half_size, frustum_size],  # 左下
        [half_size, -half_size, frustum_size],   # 右下
        [half_size, half_size, frustum_size],    # 右上
        [-half_size, half_size, frustum_size],   # 左上
    ])
    
    # 转换到世界坐标系
    corners_world = (R @ corners_cam.T).T + t
    
    # 绘制视锥边
    cam_center = corners_world[0]
    for i in range(1, 5):
        ax.plot([cam_center[0], corners_world[i, 0]],
               [cam_center[1], corners_world[i, 1]],
               [cam_center[2], corners_world[i, 2]],
               color=color, linewidth=1.5, alpha=0.7)
    
    # 绘制视锥底面
    for i in range(1, 5):
        j = i % 4 + 1
        ax.plot([corners_world[i, 0], corners_world[j, 0]],
               [corners_world[i, 1], corners_world[j, 1]],
               [corners_world[i, 2], corners_world[j, 2]],
               color=color, linewidth=2, alpha=0.9)
    
    # 绘制相机中心点
    ax.scatter(*cam_center, color=color, s=50, alpha=1.0)

# 可视化相机位姿
def visualize_camera_poses(extrinsics, positions):
    """
    可视化多个相机的位姿和轨迹
    """
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    num_views = len(extrinsics)
    colors = plt.cm.rainbow(np.linspace(0, 1, num_views))
    
    # 绘制每个相机的视锥
    for i, (extrinsic, color) in enumerate(zip(extrinsics, colors)):
        draw_camera_frustum(ax, extrinsic, fov_deg=60, frustum_size=1.5, color=color)
    
    # 绘制相机轨迹
    ax.plot(positions[:, 0], positions[:, 1], positions[:, 2],
           'k--', linewidth=2, alpha=0.5, label='Camera Trajectory')
    
    # 绘制原点 (观察目标)
    ax.scatter(0, 0, 0, color='red', s=200, marker='*', 
              label='Scene Center', alpha=0.8)
    
    # 设置坐标轴
    ax.set_xlabel('X', fontsize=12, fontweight='bold')
    ax.set_ylabel('Y', fontsize=12, fontweight='bold')
    ax.set_zlabel('Z', fontsize=12, fontweight='bold')
    ax.set_title('Camera Pose Visualization', fontsize=14, fontweight='bold', pad=20)
    ax.legend(fontsize=10)
    
    # 设置相同的坐标轴范围
    max_range = 6
    ax.set_xlim([-max_range, max_range])
    ax.set_ylim([-max_range, max_range])
    ax.set_zlim([-1, max_range])
    
    plt.tight_layout()
    plt.show()

# 生成并可视化相机位姿
extrinsics, positions = generate_camera_poses(num_views=8, radius=5.0)
visualize_camera_poses(extrinsics, positions)

print(f"Generated {len(extrinsics)} camera poses")
print(f"Extrinsics shape: {extrinsics.shape}")
print(f"Camera positions shape: {positions.shape}")

## 5. Visualizing Camera Intrinsics

VGGT 预测每个相机的内参，而不需要共享的标定：

- **FoV Prediction**: `[B, S, 2]` → `[fov_x, fov_y]`
- **Focal Length**: 从 FoV 计算
- **Per-camera Intrinsics**: 每个视图可以有不同的内参

我们将可视化不同相机的 FoV 差异。

In [ ]:
# 生成相机内参
def generate_camera_intrinsics(num_views=8):
    """
    生成每个相机的内参 (FoV)
    
    Returns:
        fov: [S, 2] FoV in degrees [fov_x, fov_y]
    """
    # 模拟不同相机的 FoV
    base_fov = 60.0
    fov_variation = 15.0
    
    fov = np.zeros((num_views, 2))
    for i in range(num_views):
        fov[i, 0] = base_fov + np.random.randn() * fov_variation  # fov_x
        fov[i, 1] = base_fov + np.random.randn() * fov_variation  # fov_y
    
    # 确保 FoV 在合理范围内
    fov = np.clip(fov, 30, 90)
    
    return fov

# FoV 转焦距
def fov_to_focal_length(fov_deg, image_size):
    """
    将 FoV (度) 转换为焦距 (像素)
    
    Args:
        fov_deg: FoV in degrees
        image_size: 图像尺寸 (宽度或高度)
    
    Returns:
        focal_length: 焦距 (像素)
    """
    fov_rad = np.deg2rad(fov_deg)
    focal_length = image_size / (2 * np.tan(fov_rad / 2))
    return focal_length

# 可视化 FoV 差异
def visualize_camera_intrinsics(fov, image_size=512):
    """
    可视化不同相机的 FoV 和焦距
    """
    num_views = len(fov)
    
    # 计算焦距
    focal_x = fov_to_focal_length(fov[:, 0], image_size)
    focal_y = fov_to_focal_length(fov[:, 1], image_size)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    view_ids = np.arange(num_views)
    
    # 1. FoV X
    axes[0, 0].bar(view_ids, fov[:, 0], color='skyblue', edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('View ID', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('FoV X (degrees)', fontsize=11, fontweight='bold')
    axes[0, 0].set_title('Horizontal Field of View', fontsize=12, fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0, 0].axhline(y=fov[:, 0].mean(), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {fov[:, 0].mean():.1f}°')
    axes[0, 0].legend()
    
    # 2. FoV Y
    axes[0, 1].bar(view_ids, fov[:, 1], color='lightcoral', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('View ID', fontsize=11, fontweight='bold')
    axes[0, 1].set_ylabel('FoV Y (degrees)', fontsize=11, fontweight='bold')
    axes[0, 1].set_title('Vertical Field of View', fontsize=12, fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    axes[0, 1].axhline(y=fov[:, 1].mean(), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {fov[:, 1].mean():.1f}°')
    axes[0, 1].legend()
    
    # 3. Focal Length X
    axes[1, 0].bar(view_ids, focal_x, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('View ID', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Focal Length X (pixels)', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Horizontal Focal Length', fontsize=12, fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    axes[1, 0].axhline(y=focal_x.mean(), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {focal_x.mean():.1f}px')
    axes[1, 0].legend()
    
    # 4. Focal Length Y
    axes[1, 1].bar(view_ids, focal_y, color='plum', edgecolor='black', alpha=0.7)
    axes[1, 1].set_xlabel('View ID', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Focal Length Y (pixels)', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Vertical Focal Length', fontsize=12, fontweight='bold')
    axes[1, 1].grid(axis='y', alpha=0.3)
    axes[1, 1].axhline(y=focal_y.mean(), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {focal_y.mean():.1f}px')
    axes[1, 1].legend()
    
    plt.suptitle('Camera Intrinsics Visualization', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # 打印统计信息
    print("=" * 60)
    print("Camera Intrinsics Statistics")
    print("=" * 60)
    print(f"FoV X: mean={fov[:, 0].mean():.2f}°, std={fov[:, 0].std():.2f}°")
    print(f"FoV Y: mean={fov[:, 1].mean():.2f}°, std={fov[:, 1].std():.2f}°")
    print(f"Focal X: mean={focal_x.mean():.2f}px, std={focal_x.std():.2f}px")
    print(f"Focal Y: mean={focal_y.mean():.2f}px, std={focal_y.std():.2f}px")
    print("=" * 60)

# 生成并可视化相机内参
fov = generate_camera_intrinsics(num_views=8)
visualize_camera_intrinsics(fov, image_size=512)

## 6. Point Cloud Fusion & Visualization

多视图点云融合是 VGGT 的关键应用：

- **Confidence-based Filtering**: 使用中位数作为阈值
- **Multi-view Merging**: 将多个视图的点云合并
- **3D Visualization**: 使用颜色编码视图 ID

我们将创建合成的多视图点云并进行融合。

In [ ]:
# 创建合成点云
def create_synthetic_point_clouds(num_views=4, points_per_view=500):
    """
    创建合成的多视图点云
    
    Returns:
        point_clouds: list of [N, 3] point clouds
        confidences: list of [N] confidence scores
        colors: list of [N, 3] colors
    """
    point_clouds = []
    confidences = []
    colors = []
    
    for i in range(num_views):
        # 创建不同区域的点云
        if i % 4 == 0:
            # 球面
            theta = np.random.rand(points_per_view) * 2 * np.pi
            phi = np.random.rand(points_per_view) * np.pi
            r = 2.0 + np.random.randn(points_per_view) * 0.1
            x = r * np.sin(phi) * np.cos(theta)
            y = r * np.sin(phi) * np.sin(theta)
            z = r * np.cos(phi)
        elif i % 4 == 1:
            # 平面
            x = np.random.randn(points_per_view) * 2
            y = np.random.randn(points_per_view) * 2
            z = np.ones(points_per_view) * 3.0 + np.random.randn(points_per_view) * 0.1
        elif i % 4 == 2:
            # 圆柱
            theta = np.random.rand(points_per_view) * 2 * np.pi
            r = 1.5
            x = r * np.cos(theta) + np.random.randn(points_per_view) * 0.1
            y = r * np.sin(theta) + np.random.randn(points_per_view) * 0.1
            z = np.random.randn(points_per_view) * 2
        else:
            # 随机散点
            x = np.random.randn(points_per_view) * 2
            y = np.random.randn(points_per_view) * 2
            z = np.random.randn(points_per_view) * 2
        
        points = np.stack([x, y, z], axis=1)
        
        # 生成置信度 (中心点置信度高)
        distances = np.linalg.norm(points, axis=1)
        confidence = np.exp(-distances / 3.0) + np.random.rand(points_per_view) * 0.2
        confidence = np.clip(confidence, 0, 1)
        
        # 为每个视图分配颜色
        color = plt.cm.rainbow(i / num_views)
        point_colors = np.tile(color[:3], (points_per_view, 1))
        
        point_clouds.append(points)
        confidences.append(confidence)
        colors.append(point_colors)
    
    return point_clouds, confidences, colors

# 点云融合
def fuse_point_clouds(point_clouds, confidences, threshold_percentile=50):
    """
    基于置信度融合多视图点云
    
    Args:
        point_clouds: list of point clouds
        confidences: list of confidence scores
        threshold_percentile: 置信度阈值百分位数
    
    Returns:
        fused_points: [N, 3] 融合后的点云
        fused_confidences: [N] 融合后的置信度
    """
    # 合并所有点云
    all_points = np.concatenate(point_clouds, axis=0)
    all_confidences = np.concatenate(confidences, axis=0)
    
    # 计算置信度阈值 (中位数)
    threshold = np.percentile(all_confidences, threshold_percentile)
    
    # 过滤低置信度的点
    mask = all_confidences >= threshold
    fused_points = all_points[mask]
    fused_confidences = all_confidences[mask]
    
    print(f"Confidence threshold: {threshold:.3f}")
    print(f"Original points: {len(all_points)}")
    print(f"Filtered points: {len(fused_points)} ({100 * len(fused_points) / len(all_points):.1f}%)")
    
    return fused_points, fused_confidences

# 可视化点云
def visualize_point_clouds(point_clouds, confidences, colors):
    """
    可视化多视图点云和融合结果
    """
    fig = plt.figure(figsize=(18, 6))
    
    # 1. 多视图点云 (按视图着色)
    ax1 = fig.add_subplot(131, projection='3d')
    for i, (points, color) in enumerate(zip(point_clouds, colors)):
        ax1.scatter(points[:, 0], points[:, 1], points[:, 2],
                   c=color, s=10, alpha=0.6, label=f'View {i+1}')
    ax1.set_xlabel('X', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Y', fontsize=11, fontweight='bold')
    ax1.set_zlabel('Z', fontsize=11, fontweight='bold')
    ax1.set_title('Multi-View Point Clouds', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=8)
    
    # 2. 多视图点云 (按置信度着色)
    ax2 = fig.add_subplot(132, projection='3d')
    all_points = np.concatenate(point_clouds, axis=0)
    all_confidences = np.concatenate(confidences, axis=0)
    scatter = ax2.scatter(all_points[:, 0], all_points[:, 1], all_points[:, 2],
                         c=all_confidences, cmap='viridis', s=10, alpha=0.6,
                         vmin=0, vmax=1)
    ax2.set_xlabel('X', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Y', fontsize=11, fontweight='bold')
    ax2.set_zlabel('Z', fontsize=11, fontweight='bold')
    ax2.set_title('Confidence-Colored Points', fontsize=12, fontweight='bold')
    plt.colorbar(scatter, ax=ax2, fraction=0.03, pad=0.1, label='Confidence')
    
    # 3. 融合后的点云
    ax3 = fig.add_subplot(133, projection='3d')
    fused_points, fused_confidences = fuse_point_clouds(point_clouds, confidences)
    scatter = ax3.scatter(fused_points[:, 0], fused_points[:, 1], fused_points[:, 2],
                         c=fused_confidences, cmap='plasma', s=15, alpha=0.7,
                         vmin=0, vmax=1)
    ax3.set_xlabel('X', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Y', fontsize=11, fontweight='bold')
    ax3.set_zlabel('Z', fontsize=11, fontweight='bold')
    ax3.set_title('Fused Point Cloud', fontsize=12, fontweight='bold')
    plt.colorbar(scatter, ax=ax3, fraction=0.03, pad=0.1, label='Confidence')
    
    plt.suptitle('Point Cloud Fusion & Visualization', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 创建并可视化点云
point_clouds, confidences, colors = create_synthetic_point_clouds(num_views=4, points_per_view=500)
visualize_point_clouds(point_clouds, confidences, colors)

## 7. Depth vs Point Head Comparison

VGGT 同时预测深度和 3D 点，但它们有不同的优势：

- **Depth + Camera Unprojection**:
  - 优点: 物理约束、更鲁棒
  - 缺点: 依赖相机内参

- **Direct Point Prediction**:
  - 优点: 端到端、无需内参
  - 缺点: 可能违反物理约束

我们将数值比较这两种方法。

In [ ]:
# 从深度生成点云
def depth_to_points(depth, fov_deg=60, image_size=128):
    """
    从深度图生成 3D 点云
    
    Args:
        depth: [H, W] 深度图
        fov_deg: 视场角 (度)
        image_size: 图像尺寸
    
    Returns:
        points: [H, W, 3] 3D 点云
    """
    H, W = depth.shape
    
    # 计算焦距
    focal = fov_to_focal_length(fov_deg, image_size)
    
    # 图像坐标
    u = np.arange(W)
    v = np.arange(H)
    u, v = np.meshgrid(u, v)
    
    # 中心化
    u = u - W / 2
    v = v - H / 2
    
    # 反投影到 3D
    x = u * depth / focal
    y = v * depth / focal
    z = depth
    
    points = np.stack([x, y, z], axis=-1)
    return points

# 比较两种方法
def compare_depth_vs_point():
    """
    比较深度+反投影 vs 直接点预测
    """
    print("=" * 60)
    print("Depth + Unprojection vs Direct Point Prediction")
    print("=" * 60)
    
    # 创建合成深度图
    H, W = 128, 128
    x = np.linspace(-2, 2, W)
    y = np.linspace(-2, 2, H)
    X, Y = np.meshgrid(x, y)
    depth_true = 5.0 + np.sqrt(X**2 + Y**2)  # 球面
    
    # 方法 1: 深度 + 反投影
    fov = 60.0
    points_from_depth = depth_to_points(depth_true, fov_deg=fov, image_size=512)
    
    # 方法 2: 直接点预测 (添加噪声模拟预测误差)
    points_direct = points_from_depth + np.random.randn(H, W, 3) * 0.2
    
    # 计算误差
    error = np.linalg.norm(points_from_depth - points_direct, axis=-1)
    
    print(f"\nMethod 1: Depth + Unprojection")
    print(f"  - Physical constraints: ✓")
    print(f"  - Requires intrinsics: ✓")
    print(f"  - Points shape: {points_from_depth.shape}")
    
    print(f"\nMethod 2: Direct Point Prediction")
    print(f"  - Physical constraints: ✗")
    print(f"  - Requires intrinsics: ✗")
    print(f"  - Points shape: {points_direct.shape}")
    
    print(f"\nError Statistics:")
    print(f"  - Mean error: {error.mean():.4f}")
    print(f"  - Std error: {error.std():.4f}")
    print(f"  - Max error: {error.max():.4f}")
    
    # 可视化
    fig = plt.figure(figsize=(16, 5))
    
    # 1. Depth map
    ax1 = fig.add_subplot(141)
    im1 = ax1.imshow(depth_true, cmap='viridis')
    ax1.set_title('Ground Truth Depth', fontsize=12, fontweight='bold')
    ax1.axis('off')
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    
    # 2. Points from depth (Z component)
    ax2 = fig.add_subplot(142)
    im2 = ax2.imshow(points_from_depth[:, :, 2], cmap='viridis')
    ax2.set_title('Depth + Unprojection (Z)', fontsize=12, fontweight='bold')
    ax2.axis('off')
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    
    # 3. Direct points (Z component)
    ax3 = fig.add_subplot(143)
    im3 = ax3.imshow(points_direct[:, :, 2], cmap='viridis')
    ax3.set_title('Direct Prediction (Z)', fontsize=12, fontweight='bold')
    ax3.axis('off')
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
    
    # 4. Error map
    ax4 = fig.add_subplot(144)
    im4 = ax4.imshow(error, cmap='hot')
    ax4.set_title('Prediction Error', fontsize=12, fontweight='bold')
    ax4.axis('off')
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)
    
    plt.suptitle('Depth vs Point Head Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("=" * 60)

compare_depth_vs_point()

## 8. Track Visualization

VGGT 的 Track Head 预测查询点在多帧中的轨迹：

- **Query Points**: 在第一帧选择的点
- **Tracks**: `[B, Q, S, 2]` 每个点在每帧的 2D 位置
- **Visibility**: `[B, Q, S]` 点在每帧的可见性
- **Confidence**: 跟踪置信度

我们将模拟跟踪数据并可视化轨迹。

In [ ]:
# 生成合成轨迹
def generate_synthetic_tracks(num_queries=20, num_frames=8, image_size=512):
    """
    生成合成的点跟踪轨迹
    
    Returns:
        tracks: [Q, S, 2] 轨迹位置
        visibility: [Q, S] 可见性
        confidence: [Q, S] 置信度
    """
    tracks = np.zeros((num_queries, num_frames, 2))
    visibility = np.zeros((num_queries, num_frames))
    confidence = np.zeros((num_queries, num_frames))
    
    for q in range(num_queries):
        # 初始位置 (第一帧)
        start_pos = np.random.rand(2) * image_size
        
        # 运动方向
        velocity = (np.random.rand(2) - 0.5) * 40  # 像素/帧
        
        for t in range(num_frames):
            # 当前位置
            pos = start_pos + velocity * t + np.random.randn(2) * 5
            tracks[q, t] = pos
            
            # 可见性 (出界则不可见)
            if 0 <= pos[0] < image_size and 0 <= pos[1] < image_size:
                visibility[q, t] = 1.0
                # 置信度随时间衰减
                confidence[q, t] = np.exp(-t / 5.0) + np.random.rand() * 0.2
            else:
                visibility[q, t] = 0.0
                confidence[q, t] = 0.0
    
    return tracks, visibility, confidence

# 可视化轨迹
def visualize_tracks(tracks, visibility, confidence, image_size=512):
    """
    可视化点跟踪轨迹
    """
    num_queries, num_frames, _ = tracks.shape
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    # 为每个轨迹分配颜色
    colors = plt.cm.rainbow(np.linspace(0, 1, num_queries))
    
    for t in range(num_frames):
        ax = axes[t]
        
        # 绘制图像边界
        ax.set_xlim(0, image_size)
        ax.set_ylim(image_size, 0)  # 反转 Y 轴
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        
        # 绘制所有轨迹的历史
        for q in range(num_queries):
            # 绘制历史轨迹 (淡化)
            history_mask = visibility[q, :t+1] > 0
            if history_mask.sum() > 1:
                history_pos = tracks[q, :t+1][history_mask]
                ax.plot(history_pos[:, 0], history_pos[:, 1],
                       color=colors[q], alpha=0.3, linewidth=1)
            
            # 绘制当前点
            if visibility[q, t] > 0:
                pos = tracks[q, t]
                conf = confidence[q, t]
                # 点的大小和透明度表示置信度
                ax.scatter(pos[0], pos[1], 
                          color=colors[q], 
                          s=50 + 100 * conf,
                          alpha=0.5 + 0.5 * conf,
                          edgecolors='black',
                          linewidths=1)
        
        ax.set_title(f'Frame {t+1}', fontsize=11, fontweight='bold')
        ax.set_xlabel('X (pixels)', fontsize=9)
        ax.set_ylabel('Y (pixels)', fontsize=9)
    
    plt.suptitle('Point Track Visualization', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # 统计信息
    print("=" * 60)
    print("Track Statistics")
    print("=" * 60)
    print(f"Number of queries: {num_queries}")
    print(f"Number of frames: {num_frames}")
    print(f"Average visibility: {visibility.mean():.2%}")
    print(f"Average confidence: {confidence[visibility > 0].mean():.3f}")
    
    # 计算轨迹长度
    track_lengths = []
    for q in range(num_queries):
        visible_frames = np.where(visibility[q] > 0)[0]
        if len(visible_frames) > 1:
            length = 0
            for i in range(len(visible_frames) - 1):
                f1, f2 = visible_frames[i], visible_frames[i+1]
                length += np.linalg.norm(tracks[q, f2] - tracks[q, f1])
            track_lengths.append(length)
    
    if track_lengths:
        print(f"Average track length: {np.mean(track_lengths):.2f} pixels")
    print("=" * 60)

# 生成并可视化轨迹
tracks, visibility, confidence = generate_synthetic_tracks(num_queries=20, num_frames=8, image_size=512)
visualize_tracks(tracks, visibility, confidence, image_size=512)

## 9. Benchmark: Speed & Memory

VGGT 在 H100 GPU 上的性能基准：

| Image Count | Resolution | Inference Time | GPU Memory (bfloat16) |
|-------------|-----------|----------------|----------------------|
| 2 views     | 512×512   | 0.15s          | 8 GB                 |
| 4 views     | 512×512   | 0.28s          | 12 GB                |
| 8 views     | 512×512   | 0.52s          | 20 GB                |
| 16 views    | 512×512   | 1.05s          | 36 GB                |

与 DUSt3R 的比较：
- **Speed**: VGGT 比 DUSt3R 快 ~2-3x
- **Memory**: VGGT 比 DUSt3R 节省 ~30% 内存
- **Accuracy**: VGGT 在多视图场景下更准确

In [ ]:
# 基准测试数据
benchmark_data = {
    'num_views': [2, 4, 8, 16],
    'vggt_time': [0.15, 0.28, 0.52, 1.05],  # seconds
    'dust3r_time': [0.35, 0.68, 1.35, 2.80],  # seconds
    'vggt_memory': [8, 12, 20, 36],  # GB
    'dust3r_memory': [11, 17, 29, 52],  # GB
}

# 创建 DataFrame
df = pd.DataFrame(benchmark_data)

# 可视化基准测试结果
def visualize_benchmark(df):
    """
    可视化 VGGT vs DUSt3R 的性能基准
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Inference Time Comparison
    ax1 = axes[0, 0]
    x = np.arange(len(df))
    width = 0.35
    ax1.bar(x - width/2, df['vggt_time'], width, label='VGGT', 
           color='skyblue', edgecolor='black', alpha=0.8)
    ax1.bar(x + width/2, df['dust3r_time'], width, label='DUSt3R', 
           color='lightcoral', edgecolor='black', alpha=0.8)
    ax1.set_xlabel('Number of Views', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Inference Time (seconds)', fontsize=11, fontweight='bold')
    ax1.set_title('Inference Speed Comparison', fontsize=12, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(df['num_views'])
    ax1.legend(fontsize=10)
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. GPU Memory Comparison
    ax2 = axes[0, 1]
    ax2.bar(x - width/2, df['vggt_memory'], width, label='VGGT', 
           color='lightgreen', edgecolor='black', alpha=0.8)
    ax2.bar(x + width/2, df['dust3r_memory'], width, label='DUSt3R', 
           color='plum', edgecolor='black', alpha=0.8)
    ax2.set_xlabel('Number of Views', fontsize=11, fontweight='bold')
    ax2.set_ylabel('GPU Memory (GB)', fontsize=11, fontweight='bold')
    ax2.set_title('GPU Memory Usage Comparison', fontsize=12, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(df['num_views'])
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)
    
    # 3. Speedup Factor
    ax3 = axes[1, 0]
    speedup = df['dust3r_time'] / df['vggt_time']
    ax3.plot(df['num_views'], speedup, marker='o', linewidth=2, 
            markersize=10, color='green', label='Speedup Factor')
    ax3.axhline(y=1.0, color='red', linestyle='--', linewidth=2, 
               label='Baseline (1x)', alpha=0.7)
    ax3.fill_between(df['num_views'], 1, speedup, alpha=0.3, color='green')
    ax3.set_xlabel('Number of Views', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Speedup Factor', fontsize=11, fontweight='bold')
    ax3.set_title('VGGT Speedup over DUSt3R', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)
    
    # 4. Memory Savings
    ax4 = axes[1, 1]
    memory_savings = (df['dust3r_memory'] - df['vggt_memory']) / df['dust3r_memory'] * 100
    ax4.plot(df['num_views'], memory_savings, marker='s', linewidth=2, 
            markersize=10, color='purple', label='Memory Savings')
    ax4.fill_between(df['num_views'], 0, memory_savings, alpha=0.3, color='purple')
    ax4.set_xlabel('Number of Views', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Memory Savings (%)', fontsize=11, fontweight='bold')
    ax4.set_title('VGGT Memory Savings over DUSt3R', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3)
    
    plt.suptitle('VGGT vs DUSt3R Benchmark (H100 GPU)', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # 打印统计信息
    print("=" * 60)
    print("Benchmark Summary")
    print("=" * 60)
    print(f"Average speedup: {speedup.mean():.2f}x")
    print(f"Max speedup: {speedup.max():.2f}x (at {df.loc[speedup.idxmax(), 'num_views']} views)")
    print(f"Average memory savings: {memory_savings.mean():.1f}%")
    print(f"Max memory savings: {memory_savings.max():.1f}% (at {df.loc[memory_savings.idxmax(), 'num_views']} views)")
    print("=" * 60)

# 可视化基准测试
visualize_benchmark(df)

# 显示表格
print("\nDetailed Benchmark Table:")
print(df.to_string(index=False))

## 10. Summary

在本 Notebook 中，我们学习了 VGGT 的推理和可视化：

### Key Takeaways

1. **Inference Pipeline**
   - 完整的端到端流程: Images → Encoder → Aggregator → Heads
   - 支持 bfloat16/float16 混合精度推理
   - GPU 内存需求随视图数量线性增长

2. **Depth Visualization**
   - 深度图和置信度的多种配色方案
   - exp 激活确保深度为正值
   - 置信度用于后处理过滤

3. **Camera Pose Visualization**
   - 相机视锥的 3D 可视化
   - 轨迹显示相机运动
   - 外参矩阵 [R|t] 编码位置和方向

4. **Camera Intrinsics**
   - 每个相机预测独立的 FoV
   - 无需共享标定
   - FoV → 焦距转换

5. **Point Cloud Fusion**
   - 多视图点云合并
   - 基于置信度的过滤 (中位数阈值)
   - 3D 散点图可视化

6. **Depth vs Point Head**
   - 深度+反投影: 物理约束、更鲁棒
   - 直接点预测: 端到端、无需内参
   - VGGT 同时使用两者以获得互补优势

7. **Track Visualization**
   - 查询点在多帧中的轨迹
   - 可见性和置信度编码
   - 运动场估计

8. **Performance Benchmark**
   - VGGT 比 DUSt3R 快 ~2-3x
   - 节省 ~30% GPU 内存
   - 更适合多视图场景

### Next Steps

- **Phase 5.8**: 实际数据集的推理和评估
- **Phase 5.9**: 3D Gaussian Splatting 集成
- **Phase 5.10**: 端到端场景重建流程

恭喜完成 Phase 5 的可视化部分！🎉